# 第五课:Attention 的一般形式——一次"可微的查表"

讲义:[05-attention.md](05-attention.md)。

## 1. softmax = 软 argmax(讲义 §3)

In [ ]:
import numpy as np

def softmax(x):
    m = np.max(x)
    e = np.exp(x - m)
    return e / e.sum()

logits = np.array([1.2, 4.5, 2.1, 3.8, 0.5])
print('hard(argmax):', np.eye(5)[np.argmax(logits)].astype(int))
print('softmax:     ', np.round(softmax(logits), 3))

## 2. 通用注意力机器:attention(q, K, V, score_fn)(讲义 §5)

In [ ]:
def attention(q, K, V, score_fn):
    scores  = np.array([score_fn(q, k) for k in K])   # 打分(第3/4课)
    weights = softmax(scores)                          # 变权重(第1课)
    return weights @ V, weights                        # 加权混合(第2课)

dot_fn    = lambda q, k: np.dot(q, k)
scaled_fn = lambda q, k: np.dot(q, k) / np.sqrt(len(q))
cos_fn    = lambda q, k: np.dot(q, k) / (np.linalg.norm(q) * np.linalg.norm(k))

X = np.array([[1.0, 1.0, 0.0, 0.0],
              [0.9, 1.0, 0.0, 0.0],
              [0.0, 0.0, 1.0, 1.0]])
words = ['小猫', '它', '可爱']
q = X[1]      # 用「它」去检索

In [ ]:
for name, fn in [('点积', dot_fn), ('缩放点积', scaled_fn), ('cosine', cos_fn)]:
    out, w = attention(q, X, X, fn)
    print(f'{name:5s} w = {np.round(w, 3)}   out = {np.round(out, 3)}')

## 3. 观察:打分函数改变权重的"尖锐程度"

- 点积(不缩放):分数差距大 → softmax 尖锐 → 前两名吃掉几乎全部预算
- 缩放点积:分数除以 √d → 差距压回 → softmax 平滑
- cosine:把范数除掉 → 只按方向比

第六课会用 d=64 的随机向量实验证明:维度一大,**不缩放就饱和**。

## 作业现场

- C2:换 cosine 后解释 w 变化——先打印三个玩具向量的范数。
- C3:批量版 attention_batch。
- E1:sparse attention 雏形(softmax 前 top-k 截断)。

In [ ]:
# TODO(C2): 观察范数被 cosine 抹掉的事实
for w_, v in zip(words, X):
    print(w_, '范数 =', round(np.linalg.norm(v), 4))

# TODO(C3): 批量版 —— Q 每行是一个 query,逐行调用 attention,返回 [n, d]
def attention_batch(Q, K, V, score_fn=scaled_fn):
    # 你的代码
    ...

# 期望「它」那行 w = [0.427 0.408 0.165]  out = [0.794 0.835 0.165 0.165]

# TODO(E1): sparse 打分 —— 保留前 2 大分数,其余置 -inf 再 softmax
def sparse_attention(q, K, V, keep=2):
    # 你的代码(提示:np.argsort 找前 keep 个,其余位置填 -np.inf,再走 softmax)
    ...